# Index Corporate Actions Report

This notebook scans the **Corporate Communications package (PubT — source `DFF004`)** for every
corporate-action category and produces a single **structured corporate-actions table** for a
universe of companies.

## Workflow Overview

1. **Configure Environment**: Load API keys and initialize services
2. **Define Universe**: Read company entity IDs
3. **Search Corporate Actions**: Source-filtered (`DFF004`) topic search across all CA categories
4. **Extract Structured Events**: One structured row per discrete corporate action (no narrative)
5. **Filter to Universe**: Keep only events that actually apply to each company
6. **Output**: A structured corporate-actions table + sources

## Output Table Columns

Company | Event Date | Status | Category | Action Type | Headline | Key Terms |
Schedule Dates | Amount / Size | Currency | Counterparty | Source Doc

`Status` is constrained to a fixed set: Announced, Confirmed, Effective, Completed,
Terminated, Amended, Withdrawn, Pending Regulatory, Proposed, Other.

## Step 1: Environment Setup

Load environment variables and verify the required API keys are present.

In [11]:
import os
import sys
import json
import asyncio
from pathlib import Path
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify API keys are set
openai_key = os.getenv('OPENAI_API_KEY')
bigdata_key = os.getenv('BIGDATA_API_KEY')

print(f"OpenAI API Key: {'Set' if openai_key else 'Missing'}")
print(f"Bigdata API Key: {'Set' if bigdata_key else 'Missing'}")

if not openai_key or not bigdata_key:
    print("\nPlease create a .env file with your API keys:")
    print("   OPENAI_API_KEY=your_key_here")
    print("   BIGDATA_API_KEY=your_key_here")

OpenAI API Key: Set
Bigdata API Key: Set


## Step 2: Initialize Services

Initialize the search and report services, load the corporate-action topic set, and define the fixed `STATUSES` enum (single source of truth).

In [12]:
from services.topic_search_service import TopicSearchService
from services.report_service import ReportService
from config.topics import ALL_CA_TOPICS

# Valid status values (single source of truth). Injected into the extraction prompt
# and used to validate/normalize the output Status column.
STATUSES = [
    "Announced",
    "Confirmed",
    "Effective",
    "Completed",
    "Terminated",
    "Amended",
    "Withdrawn",
    "Pending Regulatory",
    "Proposed",
    "Other",
]

# Initialize services
topic_search_service = TopicSearchService(api_key=bigdata_key)
report_service = ReportService()

print(f"Topic Search Service: Initialized")
print(f"Report Service: Initialized (provider: {report_service.llm_service.provider_name})")
print(f"\nCorporate Action Topics loaded: {len(ALL_CA_TOPICS)} topics")
for i, topic in enumerate(ALL_CA_TOPICS, 1):
    print(f"  {i}. {topic['topic_name']}: {topic['topic_text'][:60]}...")
print(f"\nValid statuses: {', '.join(STATUSES)}")

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x112f2b4d0>


Topic Search Service: Initialized
Report Service: Initialized (provider: openai)

Corporate Action Topics loaded: 18 topics
  1. Capital Actions: {company} cash dividend special dividend declaration ex-date...
  2. Capital Actions: {company} stock split reverse split stock consolidation scri...
  3. Capital Actions: {company} share buyback repurchase program authorization ATM...
  4. M&A / Structural: {company} merger acquisition definitive agreement deal closi...
  5. M&A / Structural: {company} divestiture asset sale business unit sale spin-off...
  6. M&A / Structural: {company} joint venture strategic partnership consortium ten...
  7. Debt / Capital Structure: {company} senior notes bond debenture issuance coupon maturi...
  8. Debt / Capital Structure: {company} term loan revolver credit facility refinancing deb...
  9. Debt / Capital Structure: {company} credit rating upgrade downgrade outlook covenant a...
  10. Equity / Shareholder: {company} rights issue rights offering priva

## Step 3: Configure Analysis Parameters

Define the universe (entity IDs), the lookback window, and the `DFF004` source filter.

In [23]:
import pandas as pd

# Read entity IDs from US_5.csv (column is RP_ENTITY_ID)
df = pd.read_csv('US_500.csv')
print(f"CSV columns: {df.columns.tolist()}")

TICKERS = df['RP_ENTITY_ID'].tolist()
print("Total entity IDs:", len(TICKERS))

LOOKBACK_DAYS = 1  # Number of days to search

# Corporate Communications package (PubT) source filter
CA_SOURCE_IDS = ["DFF004"]

# Calculate date range
end_date = datetime.now(timezone.utc)
start_date = end_date - timedelta(days=LOOKBACK_DAYS)

print(f"Analysis Configuration:")
#print(f"  Entities: {', '.join(TICKERS)}")
print(f"  Period: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print(f"  Lookback: {LOOKBACK_DAYS} days")
print(f"  Source filter (PubT): {CA_SOURCE_IDS}")

CSV columns: ['RP_ENTITY_ID']
Total entity IDs: 501
Analysis Configuration:
  Period: 2026-06-09 to 2026-06-10
  Lookback: 1 days
  Source filter (PubT): ['DFF004']


## Step 4: Search Corporate Actions

Run the corporate-action topic set for each company, restricted to source `DFF004` and with **no sentiment filter** (corporate actions are frequently neutral).

In [ ]:
import time
from typing import List, Dict


async def search_ca_news_for_ticker(ticker, days, topic_search_service, custom_topics, source_ids):
    """Search corporate-action news for a single entity (source-filtered, no sentiment filter)."""
    print(f"\nSearching corporate actions for {ticker}...")
    try:
        results = await topic_search_service.search_ticker(
            ticker=ticker,
            days=days,
            custom_topics=custom_topics,
            source_ids=source_ids,
            sentiment_values=None,  # include neutral - corporate actions are often neutral
        )
        print(f"  {ticker}: Search completed ({results.get('total_results', 0)} chunks)")
        return ticker, results
    except Exception as e:
        print(f"  {ticker}: Error - {e}")
        return ticker, None


async def search_ca_news_parallel(tickers, days, max_concurrent=10):
    """Search corporate-action news for all entities in parallel."""
    semaphore = asyncio.Semaphore(max_concurrent)

    async def process_with_semaphore(ticker):
        async with semaphore:
            return await search_ca_news_for_ticker(
                ticker, days, topic_search_service, ALL_CA_TOPICS, CA_SOURCE_IDS
            )

    print(f"Searching corporate actions for {len(tickers)} entities with {max_concurrent} parallel workers...")
    tasks = [process_with_semaphore(ticker) for ticker in tickers]
    results = await asyncio.gather(*tasks, return_exceptions=True)

    all_results = {}
    for result in results:
        if isinstance(result, Exception):
            print(f"  Task failed with exception: {result}")
        elif isinstance(result, tuple) and len(result) == 2:
            ticker, ticker_results = result
            all_results[ticker] = ticker_results
    return all_results


start_time = time.time()
search_results = await search_ca_news_parallel(TICKERS, LOOKBACK_DAYS, max_concurrent=10)
elapsed_time = time.time() - start_time

print(f"\n{'='*50}")
print(f"Search complete for {len(TICKERS)} entities")
print(f"Time taken: {elapsed_time/60:.2f} minutes ({elapsed_time:.1f} seconds)")
successful = sum(1 for v in search_results.values() if v is not None)
print(f"  Successful: {successful}")
print(f"  Failed: {len(search_results) - successful}")

Persist the raw search results to disk so the extraction step can be re-run without re-querying the API.

In [15]:
# Save raw search results so they can be re-used without re-querying
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
search_results_file = output_dir / f'ca_search_results_{timestamp}.json'
with open(search_results_file, 'w') as f:
    json.dump(search_results, f, indent=2)
print(f"Saved search results: {search_results_file}")

Saved search results: output/ca_search_results_20260610_163729.json


## Step 5: Extract Structured Corporate Actions

Use the `corporate_actions_extract` prompt to pull one structured row per discrete event. Only events where the company is the subject of the action are kept (`applies_to_company == true`), and each `status` is normalized to the fixed `STATUSES` set.

Build a per-company source map (top sources by relevance) for the report's Sources section.

In [16]:
def extract_source_map(search_results: dict, top_n: int = 3) -> dict:
    """Extract top N sources (by relevance) for each company from search results."""
    company_sources = {}
    for ticker, data in search_results.items():
        if not data:
            continue
        company_name = data.get('company_name', ticker)
        topic_results = data.get('topic_results', [])

        sources_with_relevance = []
        seen_sources = set()
        for result in topic_results:
            source = result.get('source', '')
            document_url = result.get('document_url')
            relevance = result.get('relevance', 0)
            source_key = f"{source}|{document_url}"
            if source and source_key not in seen_sources:
                seen_sources.add(source_key)
                sources_with_relevance.append({
                    'source': source,
                    'document_url': document_url,
                    'relevance': relevance,
                })

        sorted_sources = sorted(sources_with_relevance, key=lambda x: x['relevance'], reverse=True)[:top_n]
        source_map = {}
        for item in sorted_sources:
            source_name = item['source']
            if source_name in source_map:
                source_name = f"{source_name} ({len([k for k in source_map if k.startswith(source_name)]) + 1})"
            source_map[source_name] = item['document_url']
        company_sources[company_name] = source_map
    return company_sources


source_map_by_company = extract_source_map(search_results, top_n=3)
print(f"Source maps extracted for {len(source_map_by_company)} companies")

Source maps extracted for 498 companies


Run the LLM extraction in parallel. The document publication date is passed to the model as `Date:` so it uses the announcement date (never a future deadline) for `event_date`. Results are filtered to applicable events and statuses are normalized.

In [ ]:
import yaml
import re


async def extract_ca_data_for_ticker(ticker, results, llm_service, system_prompt, user_template, statuses, source_map=None):
    """Extract structured corporate-action events for a single company."""
    company_name = results.get('company_name', ticker) if results else ticker
    if not results or not results.get('topic_results'):
        return {"ticker": ticker, "company_name": company_name, "events": [], "source_map": source_map or {}}

    print(f"  Extracting corporate actions for {company_name}...")

    # Assemble context from search chunks. The document timestamp ("Date:") lets the model
    # anchor event_date to the publication/announcement date rather than a forward-looking date.
    context = ""
    for topic_result in results.get('topic_results', []):
        doc_date = (topic_result.get('timestamp') or '')[:10]
        context += f"Date: {doc_date}\n"
        context += f"Source: {topic_result.get('source', 'Unknown')}\n"
        context += f"Headline: {topic_result.get('headline', 'N/A')}\n"
        context += f"Content: {topic_result.get('full_text', topic_result.get('summary', ''))}\n"
        context += f"Document URL: {topic_result.get('document_url', 'N/A')}\n\n"

    current_datetime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    user_prompt = user_template.replace('{{current_datetime}}', current_datetime)
    user_prompt = user_prompt.replace('{{company_name}}', company_name)
    user_prompt = user_prompt.replace('{{statuses}}', ", ".join(statuses))
    user_prompt = user_prompt.replace('{{report}}', context)
    full_prompt = f"{system_prompt}\n\n{user_prompt}"

    try:
        response = await llm_service.generate_content_raw(prompt=full_prompt, model=None)
        json_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', response)
        json_str = json_match.group(1).strip() if json_match else response.strip()
        events = json.loads(json_str)
        if isinstance(events, dict):
            events = events.get('events', [])

        # Keep only events that apply to this company
        applicable = [e for e in events if e.get('applies_to_company', False)]

        # Normalize status to the allowed set
        for e in applicable:
            if e.get('status') not in statuses:
                e['status'] = 'Other'

        print(f"    {company_name}: {len(applicable)} applicable events (of {len(events)} detected)")
        return {"ticker": ticker, "company_name": company_name, "events": applicable, "source_map": source_map or {}}
    except json.JSONDecodeError as e:
        print(f"    {company_name}: JSON parse error - {e}")
        return {"ticker": ticker, "company_name": company_name, "events": [], "source_map": source_map or {}}
    except Exception as e:
        print(f"    {company_name}: Error - {e}")
        return {"ticker": ticker, "company_name": company_name, "events": [], "source_map": source_map or {}}


async def extract_ca_data_parallel(search_results, source_map_by_company=None, max_concurrent=10):
    """Extract corporate-action events for all entities in parallel."""
    with open('config/prompts.yaml', 'r', encoding='utf-8') as f:
        prompts = yaml.safe_load(f)
    prompt_config = prompts.get('corporate_actions_extract', {})
    system_prompt = prompt_config.get('system_prompt', '')
    user_template = prompt_config.get('user_template', '')

    llm_service = report_service.llm_service
    semaphore = asyncio.Semaphore(max_concurrent)

    async def process_with_semaphore(ticker, results):
        async with semaphore:
            company_name = results.get('company_name', ticker) if results else ticker
            source_map = source_map_by_company.get(company_name, {}) if source_map_by_company else {}
            return await extract_ca_data_for_ticker(
                ticker, results, llm_service, system_prompt, user_template, STATUSES, source_map
            )

    print(f"Extracting corporate actions for {len(search_results)} entities...")
    print("="*60)
    tasks = [process_with_semaphore(ticker, results) for ticker, results in search_results.items()]
    results_list = await asyncio.gather(*tasks, return_exceptions=True)

    ca_results = []
    sources_by_company = {}
    for result in results_list:
        if isinstance(result, Exception):
            print(f"  Task failed: {result}")
        elif isinstance(result, dict):
            ca_results.append(result)
            if result.get('source_map'):
                sources_by_company[result['company_name']] = result['source_map']
    return ca_results, sources_by_company


start_time = time.time()
ca_results, sources_by_company = await extract_ca_data_parallel(search_results, source_map_by_company, max_concurrent=10)
elapsed_time = time.time() - start_time

total_events = sum(len(r['events']) for r in ca_results)
companies_with_events = sum(1 for r in ca_results if r['events'])
print(f"\n{'='*60}")
print(f"Corporate action extraction complete!")
print(f"  Entities processed: {len(ca_results)}")
print(f"  Companies with applicable events: {companies_with_events}")
print(f"  Total events (pre-dedup): {total_events}")
print(f"  Time taken: {elapsed_time/60:.2f} minutes ({elapsed_time:.1f} seconds)")

## Step 6: Build Structured Corporate Actions Table

Flatten events into one row per discrete corporate action. A deterministic de-duplication step drops near-identical rows (same company, date, category, action type, and amount), and any event whose `Event Date` is in the future is excluded as a likely mis-parsed forward-looking date.

In [22]:
from datetime import date

today_iso = datetime.now(timezone.utc).strftime('%Y-%m-%d')


def _event_dedup_key(company, e):
    """Identity of a corporate action for de-duplication."""
    return (
        str(company).strip().lower(),
        str(e.get('event_date', '')).strip(),
        str(e.get('category', '')).strip().lower(),
        str(e.get('action_type', '')).strip().lower(),
        str(e.get('amount_size', '')).strip().lower(),
        str(e.get('status', '')).strip().lower(),
    )


# Flatten events into one row per corporate action (with de-duplication)
rows = []
seen_keys = set()
dropped_dupes = 0
dropped_future = 0
for r in ca_results:
    company_name = r['company_name']
    for e in r['events']:
        event_date = str(e.get('event_date', '')).strip()
        # Guard: event_date must not be in the future (it is the announcement date)
        if event_date and event_date > today_iso:
            dropped_future += 1
            continue
        key = _event_dedup_key(company_name, e)
        if key in seen_keys:
            dropped_dupes += 1
            continue
        seen_keys.add(key)
        rows.append({
            "Company": company_name,
            "Event Date": event_date,
            "Status": e.get('status', 'Other'),
            "Category": e.get('category', ''),
            "Action Type": e.get('action_type', ''),
            "Headline": e.get('headline', ''),
            "Key Terms": e.get('key_terms', ''),
            "Schedule Dates": e.get('schedule_dates', ''),
            "Amount / Size": e.get('amount_size', ''),
            "Currency": e.get('currency', ''),
            "Counterparty": e.get('counterparty', ''),
            "Source Doc": e.get('source_headline', ''),
        })

print(f"De-duplication: dropped {dropped_dupes} duplicate row(s), {dropped_future} future-dated row(s)")

if rows:
    df_ca = pd.DataFrame(rows)
    # Guarantee Status is within the allowed set
    df_ca['Status'] = df_ca['Status'].where(df_ca['Status'].isin(STATUSES), 'Other')
    # Sort by event date descending (blanks last)
    df_ca = df_ca.sort_values('Event Date', ascending=False, na_position='last').reset_index(drop=True)

    print(f"Corporate Actions Report")
    print(f"   Period: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
    print(f"   Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Total events: {len(df_ca)}")
    print("="*100)
    display(df_ca)
else:
    df_ca = pd.DataFrame()
    print("No applicable corporate actions found for the universe during the specified period.")

De-duplication: dropped 0 duplicate row(s), 0 future-dated row(s)
Corporate Actions Report
   Period: 2026-06-09 to 2026-06-10
   Generated: 2026-06-10 16:47:45
   Total events: 5


,Company,Event Date,Status,Category,Action Type,Headline,Key Terms,Schedule Dates,Amount / Size,Currency,Counterparty,Source Doc
0,Danaher Corp.,2026-06-10,Completed,M&A/Structural,Acquisition,Danaher Completes Acquisition of Masimo Corpor...,,,,,Masimo Corporation,2026-06-10 Danaher Corporation - Danaher Compl...
1,Kimco Realty Corp.,2026-06-10,Proposed,Debt/Capital Structure,Exchangeable Senior Notes Offering,"Kimco Realty OP, LLC Announces Proposed Exchan...",$125.0 million net proceeds to repurchase comm...,,$125.0 million,USD,,2026-06-10 Kimco Realty Corporation - Kimco Re...
2,Rockwell Automation Inc.,2026-06-10,Announced,Capital,Common Stock Repurchase,Rockwell Automation Approves $1 Billion for Co...,$1 billion additional share repurchase authori...,,$1 billion,USD,,2026-06-10 Rockwell Automation Approves $1 Bil...
3,Rockwell Automation Inc.,2026-06-10,Announced,Capital,Quarterly Cash Dividend,Rockwell Automation Declares Common Stock Divi...,$1.38 per share,Record: 2026-08-17 / Payable: 2026-09-10,$1.38/sh,USD,,2026-06-10 Rockwell Automation Declares Common...
4,TE Connectivity PLC,2026-06-10,Announced,Capital,Quarterly Cash Dividend,TE Connectivity declares quarterly dividend,$0.78 per ordinary share,Record: 2026-08-21 / Pay: 2026-09-11,$0.78/sh,USD,,2026-06-10 TE Connectivity declares quarterly


## Step 7: Save Outputs

Save the structured table (CSV/JSON) and the per-company source map to the `output/` directory.

In [19]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

if not df_ca.empty:
    csv_file = output_dir / f'corporate_actions_{timestamp}.csv'
    df_ca.to_csv(csv_file, index=False)
    print(f"CSV saved: {csv_file}")

# Full structured results (per company, including events)
json_file = output_dir / f'corporate_actions_{timestamp}.json'
with open(json_file, 'w') as f:
    json.dump(ca_results, f, indent=2)
print(f"JSON saved: {json_file}")

# Sources only for companies that have applicable events
companies_with_events = {r['company_name'] for r in ca_results if r['events']}
sources_file = output_dir / f'corporate_actions_sources_{timestamp}.json'
with open(sources_file, 'w') as f:
    filtered_sources = {k: v for k, v in sources_by_company.items() if k in companies_with_events}
    json.dump(filtered_sources, f, indent=2)
print(f"Sources saved: {sources_file}")

CSV saved: output/corporate_actions_20260610_163734.csv
JSON saved: output/corporate_actions_20260610_163734.json
Sources saved: output/corporate_actions_sources_20260610_163734.json


## Step 8: Generate Markdown Report

Render the structured corporate-actions table as Markdown (table only, no narrative) and save it.

In [20]:
from IPython.display import Markdown, display


def generate_ca_markdown_report(df_ca, sources_by_company, start_date, end_date):
    """Render the structured corporate-actions table plus a sources section."""
    report = f"""# Corporate Actions Report

**Period:** {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}  
**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  

---

## Corporate Actions

"""
    if df_ca is not None and not df_ca.empty:
        cols = ["Company", "Event Date", "Status", "Category", "Action Type", "Headline",
                "Key Terms", "Schedule Dates", "Amount / Size", "Currency", "Counterparty", "Source Doc"]
        report += "| " + " | ".join(cols) + " |\n"
        report += "|" + "|".join(["---"] * len(cols)) + "|\n"
        for _, row in df_ca.iterrows():
            report += "| " + " | ".join(str(row.get(c, '')).replace('|', '\\|') for c in cols) + " |\n"
        report += f"\nTotal events identified: {len(df_ca)}\n"
    else:
        report += "No applicable corporate actions found during the specified period.\n"

    report += "\n---\n\n## Sources\n\n"
    companies_with_events = set(df_ca['Company'].unique()) if (df_ca is not None and not df_ca.empty) else set()
    for company_name, sources in sources_by_company.items():
        if company_name in companies_with_events and sources:
            report += f"**{company_name}**\n"
            for source_name, url in sources.items():
                report += f"  - [{source_name}]({url})\n" if url else f"  - {source_name}\n"
            report += "\n"
    return report


ca_report = generate_ca_markdown_report(df_ca, sources_by_company, start_date, end_date)
display(Markdown(ca_report))

report_file = output_dir / f'corporate_actions_report_{timestamp}.md'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(ca_report)
print(f"Report saved: {report_file}")

# Corporate Actions Report

**Period:** 2026-06-09 to 2026-06-10  
**Generated:** 2026-06-10 16:37:34  

---

## Corporate Actions

| Company | Event Date | Status | Category | Action Type | Headline | Key Terms | Schedule Dates | Amount / Size | Currency | Counterparty | Source Doc |
|---|---|---|---|---|---|---|---|---|---|---|---|
| Danaher Corp. | 2026-06-10 | Completed | M&A/Structural | Acquisition | Danaher Completes Acquisition of Masimo Corporation |  |  |  |  | Masimo Corporation | 2026-06-10 Danaher Corporation - Danaher Completes |
| Kimco Realty Corp. | 2026-06-10 | Proposed | Debt/Capital Structure | Exchangeable Senior Notes Offering | Kimco Realty OP, LLC Announces Proposed Exchangeable Senior Notes Offering | $125.0 million to repurchase shares; remaining for corporate purposes |  | $125.0 million | USD |  | 2026-06-10 Kimco Realty Corporation - Kimco Realty OP |
| Rockwell Automation Inc. | 2026-06-10 | Announced | Capital | Common Stock Buyback | Rockwell Automation authorizes $1 billion stock repurchase | $1 billion additional for share repurchase |  | $1 billion | USD |  | 2026-06-10 Rockwell Automation Approves $1 Billion |
| Rockwell Automation Inc. | 2026-06-10 | Announced | Capital | Quarterly Cash Dividend | Rockwell Automation declares $1.38 quarterly dividend | $1.38 per share | Record: 2026-08-17 / Payable: 2026-09-10 | $1.38/sh | USD |  | 2026-06-10 Rockwell Automation Approves $1 Billion |
| TE Connectivity PLC | 2026-06-10 | Announced | Capital | Quarterly Cash Dividend | TE Connectivity declares quarterly dividend | $0.78/sh | Record: 2026-08-21 / Pay: 2026-09-11 | $0.78/sh | USD |  | 2026-06-10 TE Connectivity declares quarterly dividend |

Total events identified: 5

---

## Sources

**Danaher Corp.**
  - [PubT](https://investors.danaher.com/2026-06-10-Danaher-Completes-Acquisition-of-Masimo-Corporation)

**Kimco Realty Corp.**
  - [PubT](https://investors.kimcorealty.com/news/detail/600/kimco-realty-op-llc-announces-proposed-exchangeable-senior-notes-offering)

**Rockwell Automation Inc.**
  - [PubT](https://www.rockwellautomation.com/en-us/company/news/press-releases/Rockwell-Automation-Approves-1-Billion-for-Common-Stock-Repurchase-and-Declares-Common-Stock-Dividend-20269.html)

**TE Connectivity PLC**
  - [PubT](https://investors.te.com/news-releases/press-release-details/2026/TE-Connectivity-declares-quarterly-dividend/default.aspx)



Report saved: output/corporate_actions_report_20260610_163734.md


## Summary

The workflow is complete. Outputs generated:

- `output/corporate_actions_*.csv` — structured corporate-actions table
- `output/corporate_actions_*.json` — full per-company structured events
- `output/corporate_actions_sources_*.json` — sources per company
- `output/corporate_actions_report_*.md` — Markdown table report